# Cargo Netbacks - Determining Arb Drivers through Component Analysis

Looking at the underlying components that make up our Cargo Netbacks & Arbs to understand which component is driving daily changes in the arb.

For a full explanation of how to import our Cargo Netbacks data, please refer to our Python Jupyter Notebook Code Samples:

https://www.sparkcommodities.com/api/code-examples/jupyter.html


### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

## 1. API Base Call Functions

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

__This section can remain unchanged for most Spark API users.__

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from pprint import pprint
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.  Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/introduction.html


In [ ]:
# Input the path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

# 2. Defining Functions to fetch Cargo Netbacks data 

In [ ]:
# Define the function for listing all netback reference data
def list_netbacks(access_token):

    content = do_api_get_query(
        uri="/v1.0/netbacks/reference-data/", access_token=access_token
    )

    # retrieve release dates separately
    reldates = content["data"]["staticData"]["sparkReleases"]

    # return reference data
    refdata_dict = content["data"]

    return reldates, refdata_dict

# Fetch reference data:
reldates, refdata_dict = list_netbacks(access_token)

# formatting as a Dataframe
available_df = pd.json_normalize(refdata_dict['staticData']['fobPorts'])
available_df.head(3)

In [ ]:
# Defining function to fetch netbacks data
def fetch_netback(access_token, fob_port_uuid, release=None, via_point=None, laden=None, ballast=None, percent_hire=None, start=None, end=None, format='json'):
    
    query_params = "?fob-port={}".format(fob_port_uuid)
    if release is not None:
        query_params += "&release-date={}".format(release)
    if start is not None:
        query_params += "&start={}".format(start)
    if end is not None:
        query_params += "&end={}".format(end)
    if via_point is not None:
        query_params += "&via-point={}".format(via_point)
    if laden is not None:
        query_params += "&laden-congestion-days={}".format(laden)
    if ballast is not None:
        query_params += "&ballast-congestion-days={}".format(ballast)
    if percent_hire is not None:
        query_params += "&percent-hire={}".format(percent_hire)
    
    
    content = do_api_get_query(
        uri="/v1.0/netbacks/{}".format(query_params),
        access_token=access_token, format=format,
    )
    
    if format == 'json':
        data = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            data = content
            print('No Data to Show')
        else:
            data = content.decode('utf-8')
            data = pd.read_csv(StringIO(data))  # automatically converting into a Pandas DataFrame when choosing CSV format
            data['ReleaseDate'] = pd.to_datetime(data['ReleaseDate'])
            data['LoadDate'] = pd.to_datetime(data['LoadDate'])

    return data


## N.B. Historical Data Limits

Currently, a maximum of 1 year's worth of historical data can be called at one time due to the size of the data file. 

If more data points are required, the below code can be used: the function calls the historical data 1 year at a time and combines the data into one Pandas DataFrame

In [ ]:

def loop_historical_data(access_token, fob_port_uuid=None, via_point=None, percent_hire=None, start=None, end=None):
    
    hist_diff = (datetime.datetime.strptime(end, '%Y-%m-%d') - datetime.datetime.strptime(start, '%Y-%m-%d')).days
    t = 0

    starts = []
    ends = []

    w = 365

    while t < hist_diff:
        # initialising dataframe
        if t == 0 and hist_diff>w:
            diff_end = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
            hist_df = fetch_netback(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point, start=start, end=diff_end, 
                                    percent_hire=percent_hire, format='csv')

            starts.append(start)
            ends.append(diff_end)
        
        elif t==0 and hist_diff<=w:
            hist_df = fetch_netback(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point, start=start, end=end, 
                                    percent_hire=percent_hire, format='csv')

        # appending additional historical data
        else:
            if t < hist_diff-w:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=w), '%Y-%m-%d')
                historical_addition = fetch_netback(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point, start=diff_start, end=diff_end, 
                                    percent_hire=percent_hire, format='csv')
                
                try:
                    hist_df = pd.concat([hist_df,historical_addition])
                #exception if hist_df is empty
                except:
                    hist_df = historical_addition.copy()
                starts.append(start)
                ends.append(diff_end)
            else:
                diff_start = datetime.datetime.strftime(datetime.datetime.strptime(start, '%Y-%m-%d') + pd.Timedelta(days=t+1), '%Y-%m-%d')
                diff_end = datetime.datetime.strftime(datetime.datetime.strptime(diff_start, '%Y-%m-%d') + pd.Timedelta(days=hist_diff-t), '%Y-%m-%d')

                historical_addition = fetch_netback(access_token, fob_port_uuid=fob_port_uuid, via_point=via_point, start=diff_start, end=diff_end, 
                                    percent_hire=percent_hire, format='csv')

                hist_df = pd.concat([hist_df, historical_addition])
                starts.append(start)
                ends.append(diff_end)
        
        #looping by year
        t += w

    for c in list(hist_df.columns)[4:]:
        hist_df[c] = pd.to_numeric(hist_df[c])
    hist_df['ReleaseDate'] = pd.to_datetime(hist_df['ReleaseDate'])
    hist_df['LoadMonthDate'] = pd.to_datetime(hist_df['LoadMonth'])
    hist_df['CalMonth'] = hist_df['LoadMonthDate'].dt.strftime('%b%y')

    hist_df = hist_df.drop_duplicates()
    hist_df = hist_df.sort_values(['ReleaseDate', 'LoadDate'])

    return hist_df

# 3. Defining Inputs & Calling Data

In [ ]:
# Defining date-range for data call
start = '2026-03-01'
end = datetime.datetime.today().strftime('%Y-%m-%d')

# Fetching port & via point inputs
sab = available_df[available_df["name"] == 'Sabine Pass']["uuid"].iloc[0]

# Fetching data with defined input params
df_csv = loop_historical_data(access_token, start=start, end=end, fob_port_uuid=sab, via_point='cogh')

# Converting Freight columns to negative to indicate them as costs
df_csv['NeaMetaRouteCost'] = -df_csv['NeaMetaRouteCost']
df_csv['NweMetaRouteCost'] = -df_csv['NweMetaRouteCost']

df_csv

In [ ]:
# Calculating Arb Components from underlying Netbacks components: DES LNG pricing
df_csv['DeltaMetaDesLng'] = df_csv['NeaMetaDesLng'] - df_csv['NweMetaDesLng']
df_csv['DeltaMetaTtfPrice'] = df_csv['NeaMetaTtfPrice'] - df_csv['NweMetaTtfPrice']
df_csv['DeltaMetaTtfBasis'] = (df_csv['NeaMetaTtfBasis'] - df_csv['NweMetaTtfBasis'])

# Calculating Arb Components from underlying Netbacks components: Freight costs
df_csv['DeltaMetaRouteCost'] = (df_csv['NeaMetaRouteCost'] - df_csv['NweMetaRouteCost'])
df_csv['DeltaMetaVolAdj'] = df_csv['NeaMetaVolAdj'] - df_csv['NweMetaVolAdj']
df_csv['DeltaFreight'] = df_csv['DeltaMetaRouteCost'] + df_csv['DeltaMetaVolAdj']

In [ ]:
# Creating Front Month dataframe
m1 = df_csv[df_csv['LoadMonthIndex'] == 1].copy()

# make sure data is sorted
m1 = m1.sort_values('ReleaseDate', ascending=False)

# Creating day-on-day diffs for each component
m1['DiffDesLng'] = m1['DeltaMetaDesLng'].diff(periods=-1)
m1['DiffTtfPrice'] = m1['DeltaMetaTtfPrice'].diff(periods=-1)
m1['DiffTtfBasis'] = m1['DeltaMetaTtfBasis'].diff(periods=-1)
m1['DiffRouteCost'] = m1['DeltaMetaRouteCost'].diff(periods=-1)
m1['DiffVolAdj'] = m1['DeltaMetaVolAdj'].diff(periods=-1)

m1.head(3)

# 4. Plotting

In [ ]:
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

# Component naming & colour mapping
comp_cols = {
    'DiffTtfPrice':  'TTF Price',
    'DiffTtfBasis':  'TTF Basis',
    'DiffRouteCost': 'Route Cost',
    'DiffVolAdj':    'Vol Adj',
}
comp_colours = {
    'TTF Price':  '#008bcc',
    'TTF Basis':  '#9B59B6',
    'Route Cost': '#1E8449',
    'Vol Adj':    '#52BE80',
}

filt_threshold = 0.03    # grey out days whose net move is below this ($/MMBtu)
hide_weekend = False    # set to True if you want to exclude weekends (i.e. set to white) from "dominant component" graph

# Creating function to stack positive components and negative components separately
def stacked_signed_bars(ax, x, df, cols, colours, width=0.8):
    pos = np.zeros(len(x))
    neg = np.zeros(len(x))
    handles = []
    for col, label in cols.items():
        v = df[col].fillna(0).to_numpy(float)
        bottoms = np.where(v >= 0, pos, neg)
        h = ax.bar(x, v, bottom=bottoms, width=width, color=colours[label],
                   label=label, edgecolor='white', linewidth=0.4, zorder=1)
        pos = pos + np.where(v >= 0, v, 0.0)
        neg = neg + np.where(v <  0, v, 0.0)
        handles.append(h)
    return handles

#
net = m1[list(comp_cols)].sum(axis=1)  # Daily diff in arb
signed = m1[list(comp_cols)].mul(np.sign(net), axis=0)  # calc direction of effect of each component
valid = signed.notna().any(axis=1)                       # rows with at least one non-NaN component
dom_label = signed[valid].idxmax(axis=1).map(comp_cols)  # dominant component, valid rows only
dom_label = dom_label.reindex(signed.index)              # restore all-NaN rows as NaN
dom_colour = dom_label.map(comp_colours) # fetching colour of dominant component
dom_colour[net.abs() < filt_threshold] = '#dddddd' # aplying filter threshold
dom_colour = dom_colour.fillna('#dddddd') # applying grey to dates with no dominant component

# Figure: main chart on top, dominance strip below (shared x)
fig, (ax, ax_strip) = plt.subplots(
    2, 1, figsize=(14, 7.6), sharex=True,
    gridspec_kw={'height_ratios': [14, 1], 'hspace': 0.10}
)

# Bars on a twin y-axis, arb line on the primary axis
# remove the first line below, and change all "axb" values to "ax", to plot on same y-axis
axb = ax.twinx()
bar_handles = stacked_signed_bars(axb, m1['ReleaseDate'], m1, comp_cols, comp_colours)
axb.axhline(0, color='grey', linewidth=0.8, zorder=0)
axb.set_ylabel('Daily change in arb components ($/MMBtu)')
axb.grid(False)

line, = ax.plot(m1['ReleaseDate'], m1['DeltaNeaNwe'], color='#1A3A6B',
                label='US Arb (NEA via COGH)', linewidth=3.0, zorder=3, alpha=0.5)
ax.scatter(m1['ReleaseDate'].iloc[0], m1['DeltaNeaNwe'].iloc[0],
           color='#1A3A6B', marker='o', s=80, zorder=4, alpha=0.5)
ax.set_ylabel('Arb level ($/MMBtu)')

# Line axis in front of the bar axis
ax.set_zorder(axb.get_zorder() + 1)
ax.patch.set_visible(False)

ax.set_xlim([m1['ReleaseDate'].iloc[-1] - pd.Timedelta(1, 'day'),
             m1['ReleaseDate'].iloc[0]  + pd.Timedelta(2, 'day')])

# creating legend
handles = [line] + bar_handles
ax.legend(handles, [h.get_label() for h in handles], loc='lower right')

# ── Dominance strip ────────────────────────────────────────────────────────
# creating dataframe for the dominant component strip
strip = (pd.DataFrame({'date': m1['ReleaseDate'].to_numpy(),
                        'color': dom_colour.to_numpy()})
          .sort_values('date').reset_index(drop=True))
xnum  = mdates.date2num(strip['date'])

# calculating segment positions for each bar to align with arb chart
if hide_weekend:
    lefts  = xnum - 0.5
    widths = np.full(len(xnum), 1.0)
else:
    mid    = (xnum[:-1] + xnum[1:]) / 2
    edges  = np.concatenate([[xnum[0]  - (xnum[1]  - xnum[0])  / 2],
                              mid,
                              [xnum[-1] + (xnum[-1] - xnum[-2]) / 2]])
    lefts  = edges[:-1]
    widths = np.diff(edges)

# Plotting dominant component strip chart
ax_strip.bar(lefts, height=1.0, width=widths, bottom=0, align='edge',
             color=strip['color'].to_numpy(), linewidth=0)
ax_strip.set_ylim(0, 1)
ax_strip.set_yticks([])
ax_strip.set_ylabel('Dominant\ndriver', rotation=0, ha='right', va='center', labelpad=25)
ax_strip.set_xlabel('Release Date')
ax_strip.grid(False)

plt.tight_layout()
plt.show()